# Voz_Noslen F5-TTS ONNX "Modo Turbo" - v2026.06.16.5

Este notebook implementa a exportação End-to-End do F5-TTS para ONNX com quantização INT8.

**O que este notebook faz:**
1.  **Keep-alive**: Evita que o Kaggle desconecte por inatividade.
2.  **Criação de arquivos**: Restaura o script e dependências no worker.
3.  **Instalação**: Instala apenas os pacotes necessários.
4.  **Exportação Turbo**: Cria um Wrapper completo (Transformer + Euler + Vocos).
5.  **Quantização INT8**: Reduz o modelo de ~5GB para ~1.2GB.
6.  **Upload**: Envia o resultado para o Hugging Face Repo.

**Como rodar:**
- Ative **Internet** (Settings -> Internet On).
- Adicione o Secret `HF_TOKEN` se for fazer upload.
- Clique em **Run All**.

In [ ]:
# 0) Keep-alive do navegador (evita desconexão)
from IPython.display import display, HTML
display(HTML("""
<script>
function clickKeepAlive() {
    console.log("Kaggle Keep-alive heartbeat");
    document.querySelector("button[data-test-id='run-all-button']")?.click();
    // Simula interação leve a cada 10 minutos
}
setInterval(clickKeepAlive, 600000);
</script>
"""))

In [ ]:
# 1) Reconstrução dos Arquivos de Lógica (Base64 Safe Mode)
import base64
from pathlib import Path

script_b64 = "..."
reqs_b64 = "aHVnZ2luZ2ZhY2VfaHViPj0wLjMxLjAsPDEuMApoZi14ZXQ+PTEuMS4wCmY1LXR0cz49MS4xLjkKc2FmZXRlbnNvcnM+PTAuNC41CnRvcmNoPj0yLjQuMAp0b3JjaGF1ZGlvPj0yLjQuMApzb3VuZGZpbGU+PTAuMTIuMQpsaWJyb3NhPj0wLjEwLjEsPDAuMTEuMApweWR1Yj49MC4yNS4xCnZvY29zPj0wLjEuMApoeWRyYS1jb3JlPj0xLjMuMgpvbWVnYWNvbmY+PTIuMy4wCmNhY2hlZC1wYXRoPj0xLjUuMSw8Mi4wLjAKdHJhbnNmb3JtZXJzPj00LjM2LjAsPDUuMC4wCmFjY2VsZXJhdGU+PTAuMjUuMApncmFkaW8+PTQuNDQuMApvbm54Pj0xLjE2LjAKb25ueHNjcmlwdD49MC4xLjAKb25ueHJ1bnRpbWU+PTEuMTguMApudW1weQpzY2lweQpwYW5kYXMK"

WORK = Path("/kaggle/working")
script_path = WORK / "f5_tts_onnx_packager_kaggle.py"
reqs_path = WORK / "conversor_voz_requirements_kaggle.txt"

script_path.write_bytes(base64.b64decode(script_b64))
reqs_path.write_bytes(base64.b64decode(reqs_b64))

print(f"Script criado: {script_path.name}")
print(f"Requirements criado: {reqs_path.name}")

In [ ]:
# 2) Instalação de Dependências
# Nota: onnxruntime converter-common é necessário para otimizações de quantização
!pip install -q -r conversor_voz_requirements_kaggle.txt
!pip install -q onnxconverter-common

In [ ]:
# 3) Configurações de Ambiente
import os
from datetime import datetime, timezone
from kaggle_secrets import UserSecretsClient

try:
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
except: 
    print("AVISO: HF_TOKEN não encontrado. Downloads públicos ok, mas upload falhará.")

# Configurações padrão do Modo Turbo
os.environ["HF_SOURCE_URL"] = "https://huggingface.co/buckets/warllem/Voz_Noslen"
os.environ["HF_VOICE_DIR"] = "voices/v_minha_voz_f5_tts_ptbr"
os.environ["HF_UPLOAD_REPO_ID"] = "warllem/Voz_Noslen_ONNX"
os.environ["HF_TARGET_FOLDER"] = "onnx_packages/turbo_" + datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
os.environ["HF_DOWNLOAD_MODE"] = "essential"

print("Configurações prontas para o Modo Turbo.")

In [ ]:
# 4) Execução do Processamento End-to-End (Modo Turbo)
import subprocess, sys, time

cmd = [sys.executable, "f5_tts_onnx_packager_kaggle.py"]
print(f"Iniciando: {' '.join(cmd)}")

process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)

# Heartbeat para monitorar progresso longo
start_time = time.time()
for line in process.stdout: 
    print(f"[{time.time() - start_time:0.1f}s] {line}", end="")

rc = process.wait()
if rc == 0:
    print("\n--- SUCESSO! O pacote ONNX Turbo foi gerado e enviado. ---")
else:
    print(f"\n--- ERRO: O processamento falhou com código {rc} ---")

In [ ]:
# 5) Validação de Saída Local
from pathlib import Path
pkg_dir = Path("/kaggle/working/voz_noslen_onnx_package")
if pkg_dir.exists():
    print("Arquivos gerados no worker:")
    for f in sorted(pkg_dir.rglob("*.onnx")):
        size = f.stat().st_size / (1024*1024)
        print(f"- {f.relative_to(pkg_dir)} ({size:.2f} MB)")
else:
    print("Pasta de saída não encontrada. Verifique os logs da célula anterior.")